# Public Government Data Collection
Use only public official pages. Collection is opt-in, snapshots go to ignored `data/raw/public/`, and extracted candidates go to ignored `data/interim/`. Never access authenticated pages or bypass CAPTCHA.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import requests
import yaml
from bs4 import BeautifulSoup

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DOCUMENT_DIR = PROJECT_ROOT / 'documents'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
sys.path.insert(0, str(PROJECT_ROOT))

SOURCE_URLS = yaml.safe_load((PROJECT_ROOT / 'config' / 'sources.yaml').read_text(encoding='utf-8'))
print(f'Configured {sum(len(group) for group in SOURCE_URLS.values())} official source URLs')

In [ ]:
from src.data_collection.source_registry import flatten_sources
from src.data_processing.build_datasets import add_observed_links_to_registry

source_registry = add_observed_links_to_registry(flatten_sources(SOURCE_URLS))
source_registry.to_csv(PROCESSED_DIR / 'official_sources.csv', index=False, encoding='utf-8-sig')
source_registry[['source_id', 'url', 'verification_status', 'last_verified']]

## Optional snapshots
Set `COLLECT_PUBLIC_PAGES = True` only when you intend to fetch the listed public pages. This saves raw HTML and review-only table candidates. It does not download PDFs or copy extracted data into processed datasets.

In [ ]:
from src.data_collection.meeseva_scraper import collect_meeseva_candidates
from src.data_collection.services_scraper import save_public_snapshot

COLLECT_PUBLIC_PAGES = False
if COLLECT_PUBLIC_PAGES:
    state_url = SOURCE_URLS['telangana']['state_services']
    forms_url = SOURCE_URLS['telangana']['forms']
    state_snapshot = save_public_snapshot(state_url, 'telangana_state_services')
    forms_snapshot = save_public_snapshot(forms_url, 'telangana_forms')
    meeseva_candidates = collect_meeseva_candidates()
    print(state_snapshot, forms_snapshot, meeseva_candidates)
else:
    print('No network requests made. Review source registry before enabling collection.')

## Forms and provenance
Form metadata belongs in `documents_and_forms.csv`, with the official download link, listing-page `source_url`, and review date. Keep downloaded originals under `documents/`; do not commit PDFs. Source presence alone does not verify eligibility, fees, processing times, or active status.